In [61]:
import pandas as pd
import requests
import os
from datetime import datetime
from dotenv import load_dotenv
from pathlib import Path

In [62]:
load_dotenv(override=True)

API_KEY = os.getenv("OPEN_WEATHER_MAP_API_KEY")
LAT = float(os.getenv("LATITUDE"))
LON = float(os.getenv("LONGITUDE"))

In [63]:
def get_historical_weather(ts):
    unix_ts = int(ts.timestamp())
    url = f"https://api.openweathermap.org/data/3.0/onecall/timemachine?lat={LAT}&lon={LON}&dt={unix_ts}&appid={API_KEY}"
    response = requests.get(url).json()

    if "data" in response:
        weather = response["data"][0]
        return pd.Series({
            "wind_speed": weather.get('wind_speed'),
            "wind_deg": weather.get('wind_deg'),
            "pressure": weather.get('pressure')
        })
    
    return pd.Series({
        "wind_speed": None,
        "wind_deg": None,
        "pressure": None
    })

In [82]:
data = pd.read_parquet(Path("../data/data.parquet"))
data['timestamp'] = pd.to_datetime(data["period.datetime_from.utc"])
ts_sample = data.loc[:5, "timestamp"].dt.round('h')
unique_hours = pd.Series(ts_sample.unique())
unix_start = int(unique_hours[0].timestamp())
unix_end = int(unique_hours[5].timestamp())

In [65]:
url = f"https://api.openweathermap.org/data/2.5/weather?lat={LAT}&lon={LON}&appid={API_KEY}"
response = requests.get(url).json()

In [69]:
response.keys()

dict_keys(['coord', 'weather', 'base', 'main', 'visibility', 'wind', 'clouds', 'dt', 'sys', 'timezone', 'id', 'name', 'cod'])

In [84]:
url = f"https://history.openweathermap.org/data/2.5/history/city?lat={LAT}&lon={LON}&type=hour&start={unix_start}&end={unix_end}&appid={API_KEY}"
response = requests.get(url).json()

In [86]:
response

{'cod': 401,
 'message': 'Invalid API key. Please see http://openweathermap.org/faq#error401 for more info.'}